# Video huấn luyện Ultralight bằng MuseTalk 1.5 (thầy bản dev, ADR-064)

Notebook chạy **trên Kaggle**, không chạy trên laptop. MuseTalk 1.5 lồng miệng theo cả file tiếng huấn luyện (`train_v1.wav`, khoảng
4,9 phút tiếng VieNeu) lên 81 khung InfiniteTalk (version 6), phát tới rồi lùi cho đủ độ dài. Kết quả là video có tiếng mà
`train_ultralight.ipynb` dùng khi `TEACHER = "musetalk"`.

**Thầy dev và thầy prod (ADR-064):**
- `musetalk` (notebook này): nhanh, khoảng 20–30 phút trên T4 miễn phí; miệng mở ít và mờ hơn. Dùng để làm và thử cả đường
  Ultralight bây giờ.
- `infinitetalk` (`make_infinitetalk_video.ipynb`, `RUN_NAME = "train_v1"`): miệng tốt hơn nhưng rất chậm. Dùng trước buổi demo.
- Hai notebook ra cùng một kiểu đầu ra: một file mp4 có tiếng, 25 khung/giây, cùng thư mục với `run.json`. Notebook huấn luyện chỉ
  cần đổi `TEACHER`.

**Quy tắc:**
- Đây là dữ liệu huấn luyện offline, không bao giờ phát live (ràng buộc 1 của CLAUDE.md).
- Nhân vật AI `demo_host_02` chỉ dùng cho một khách, một shop.
- Video, khung hình và `run.json` **không đưa vào git**.

**Cách chạy:**
1. Dataset `jintran0501/ai-live-infinitetalk-input` phải có `train_v1.wav` (lấy ở
   `assets/avatars/demo_host_02/training/kaggle_input_train/train_v1.wav` trên laptop) và thư mục `infinitetalk_v6_frames/`.
   Chưa có thì dataset → New Version → tải `train_v1.wav` lên, giữ các file cũ.
2. Add Input → **Datasets** → dataset đó, bản mới nhất.
3. Session options: Accelerator = **GPU T4** (một GPU là đủ), Internet = **On**.
4. Save Version → Save & Run All. Kết quả ở `musetalk_out/<RUN_NAME>/` trong panel Output.

**Xem gì:** `preview.mp4` (15 giây có tiếng, sau đoạn im lặng đầu) và `mouth_sheet.png`. Video đầy đủ là
`<CHARACTER_ID>_musetalk_<RUN_NAME>.mp4`.

## 1. Cấu hình

In [ ]:
CHARACTER_ID = "demo_host_02"  # one character = one customer, one shop
TEACHER = "musetalk"  # ADR-064: the dev teacher; train_ultralight.ipynb looks for this name
RUN_NAME = "train_v1"
VIDEO_NAME = f"{CHARACTER_ID}_{TEACHER}_{RUN_NAME}.mp4"  # train_ultralight.ipynb TEACHERS["musetalk"]["video"]

# Inputs are attached in the Kaggle UI (Add Input); both are searched recursively under /kaggle/input.
AUDIO_NAME = "train_v1.wav"  # VieNeu voice reading the training script (ADR-060)
FRAME_GLOB = "frame_*.png"  # 81 InfiniteTalk frames (version 6), played forward then backward
FPS = 25  # Ultralight with HuBERT needs 25 fps

# Official MuseTalk (MIT), pinned; only the v1.5 inference path is used.
MUSETALK_REPO = "https://github.com/TMElyralab/MuseTalk"
MUSETALK_COMMIT = "0a89dec45a0192b824e3cf4daf96c239440c5ed8"

# Hugging Face revisions pinned in ADR-063.
HF_WEIGHTS = {
    "musetalk": (
        "TMElyralab/MuseTalk",
        "2bcb936e2fddb4d86db4c62fd45b387d0c061571",
        ["musetalkV15/musetalk.json", "musetalkV15/unet.pth"],
        "",
    ),
    "sd-vae": (
        "stabilityai/sd-vae-ft-mse",
        "31f26fdeee1355a5c34592e401dd41e45d25a493",
        ["config.json", "diffusion_pytorch_model.safetensors"],
        "sd-vae",
    ),
    "whisper": (
        "openai/whisper-tiny",
        "169d4a4341b33bc18d8881c4b69c2e104e1cc0af",
        ["config.json", "model.safetensors", "preprocessor_config.json"],
        "whisper",
    ),
}
FACE_PARSE_GDRIVE_ID = "154JgKpzCPW82qINcVieuPH3fZ2e0P812"  # same source as MuseTalk's download_weights.sh
FACE_PARSE_HF_FALLBACK = ("ManyOtherFunctions/face-parse-bisent", "0073b233a5a3c4b1377d4dbf49245017938a72b5")
RESNET18_URL = "https://download.pytorch.org/models/resnet18-5c106cde.pth"
RESNET18_SHA256_PREFIX = "5c106cde"

# Defaults of the official v1.5 inference script, as in the compare run (ADR-063).
BATCH_SIZE = 8
# v1.5 adds this margin below the chin to the crop, which leaves the jaw room to move.
# TODO: if the mouth still opens too little, try 15-20 and compare preview.mp4 (ADR-064).
EXTRA_MARGIN = 10
LEFT_CHEEK_WIDTH = RIGHT_CHEEK_WIDTH = 90
PARSING_MODE = "jaw"
AUDIO_PADDING_LEFT = AUDIO_PADDING_RIGHT = 2
USE_FP16 = True

# train_v1.wav starts with 20 s of silence, so the preview and the mouth sheet start after it.
PREVIEW_START_S = 20.0
PREVIEW_SECONDS = 15.0
MOUTH_CLOSEUP_PX = 360

## 2. Môi trường và mã MuseTalk

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import tarfile
import time
from datetime import UTC, datetime
from pathlib import Path

import torch  # preinstalled by Kaggle with the right CUDA build

ON_KAGGLE = Path("/kaggle").exists()
if not ON_KAGGLE:
    raise SystemExit("Notebook này chỉ chạy trên Kaggle (ADR-063, ADR-064), không chạy trên laptop.")
if not torch.cuda.is_available():
    raise SystemExit("Chưa bật GPU: Session options → Accelerator = GPU T4.")

MUSETALK_DIR = Path("/tmp/MuseTalk")
WORK_DIR = Path("/tmp/musetalk_work")
OUT_DIR = Path("/kaggle/working/musetalk_out") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# transformers 5.x (Kaggle default) changed how encoders return hidden states, which MuseTalk's whisper path
# relies on; 4.57.6 is the last 4.x and matches the InfiniteTalk notebook. diffusers 0.30.2 is the official pin.
PIP_PINS = [
    "transformers==4.57.6",
    "diffusers==0.30.2",
    "face-alignment==1.4.1",
    "gdown",
    "einops",
    "omegaconf",
    "librosa",
]


def run(*cmd: str, cwd: Path | None = None) -> str:
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} lỗi:\n{result.stderr[-3000:]}")
    return result.stdout.strip()


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


if not MUSETALK_DIR.exists():
    MUSETALK_DIR.mkdir(parents=True)
    run("git", "init", "-q", cwd=MUSETALK_DIR)
    run("git", "fetch", "-q", "--depth", "1", MUSETALK_REPO, MUSETALK_COMMIT, cwd=MUSETALK_DIR)
    run("git", "checkout", "-q", "FETCH_HEAD", cwd=MUSETALK_DIR)
assert run("git", "rev-parse", "HEAD", cwd=MUSETALK_DIR) == MUSETALK_COMMIT

# Keep Kaggle's CUDA build of torch: none of the pins below pull torch.
run(sys.executable, "-m", "pip", "install", "-q", *PIP_PINS)
run("ffmpeg", "-version")

import diffusers  # noqa: E402
import face_alignment  # noqa: E402
import transformers  # noqa: E402

GPU_NAME = torch.cuda.get_device_name(0)
print(
    f"GPU: {GPU_NAME} · torch {torch.__version__} · transformers {transformers.__version__} · "
    f"diffusers {diffusers.__version__} · face-alignment {face_alignment.__version__} · "
    f"MuseTalk {MUSETALK_COMMIT[:10]}"
)

## 3. Trọng số (ghim revision, ghi sha256)

In [ ]:
from huggingface_hub import hf_hub_download  # noqa: E402

MODELS_DIR = MUSETALK_DIR / "models"
weights_info: dict[str, dict] = {}

for key, (repo, revision, files, subdir) in HF_WEIGHTS.items():
    target = MODELS_DIR / subdir if subdir else MODELS_DIR
    for name in files:
        started = time.time()
        path = Path(hf_hub_download(repo, name, revision=revision, local_dir=target))
        weights_info[f"{key}/{name}"] = {"repo": repo, "revision": revision, "sha256": sha256(path)}
        print(f"{repo}/{name} @ {revision[:10]} ({time.time() - started:.0f} s)")

face_parse_dir = MODELS_DIR / "face-parse-bisent"
face_parse_dir.mkdir(parents=True, exist_ok=True)
face_parse_path = face_parse_dir / "79999_iter.pth"
face_parse_source = f"gdrive:{FACE_PARSE_GDRIVE_ID}"
if not face_parse_path.exists():
    try:
        run(sys.executable, "-m", "gdown", FACE_PARSE_GDRIVE_ID, "-O", str(face_parse_path))
    except RuntimeError as error:
        # Google Drive often rate-limits shared files; fall back to the pinned HF copy.
        print(f"gdown lỗi, dùng bản sao HF: {str(error)[-300:]}")
        repo, revision = FACE_PARSE_HF_FALLBACK
        hf_hub_download(repo, "79999_iter.pth", revision=revision, local_dir=face_parse_dir)
        face_parse_source = f"hf:{repo}@{revision}"
weights_info["face-parse/79999_iter.pth"] = {"source": face_parse_source, "sha256": sha256(face_parse_path)}

resnet_path = face_parse_dir / "resnet18-5c106cde.pth"
if not resnet_path.exists():
    # torchvision names weight files after their sha256 prefix; download_url_to_file rejects a mismatch.
    torch.hub.download_url_to_file(RESNET18_URL, str(resnet_path), hash_prefix=RESNET18_SHA256_PREFIX)
weights_info["face-parse/resnet18-5c106cde.pth"] = {"source": RESNET18_URL, "sha256": sha256(resnet_path)}

# The 2017 torchvision ResNet-18 file uses the legacy tar format, which torch>=2.6 cannot open with weights_only and
# which MuseTalk's FaceParsing loads with torch's default (weights_only=True). Its sha256 prefix is checked above and
# it comes from download.pytorch.org, so it is unpickled once and re-saved as a plain state dict in the zip format.
if tarfile.is_tarfile(resnet_path):
    state_dict = torch.load(resnet_path, map_location="cpu", weights_only=False)
    torch.save({key: value for key, value in state_dict.items()}, resnet_path)
    weights_info["face-parse/resnet18-5c106cde.pth"]["converted_sha256"] = sha256(resnet_path)
    print("resnet18-5c106cde.pth: định dạng tar cũ, đã lưu lại dạng zip (state dict).")

# Every other checkpoint must be a plain state dict: refuse anything that needs pickle code execution.
for path in (face_parse_path, resnet_path, MODELS_DIR / "musetalkV15" / "unet.pth"):
    torch.load(path, map_location="cpu", weights_only=True)
print("Đủ trọng số; các file .pth nạp được ở chế độ weights_only.")

## 4. Đầu vào: 81 khung InfiniteTalk và file tiếng huấn luyện

In [ ]:
import cv2  # noqa: E402
import numpy as np  # noqa: E402

INPUT_ROOT = Path("/kaggle/input")
frame_paths = sorted(INPUT_ROOT.rglob(FRAME_GLOB))
audio_candidates = sorted(INPUT_ROOT.rglob(AUDIO_NAME))
if not frame_paths:
    raise SystemExit(f"Không thấy {FRAME_GLOB}: thiếu thư mục infinitetalk_v6_frames/ trong dataset.")
if not audio_candidates:
    raise SystemExit(
        f"Không thấy {AUDIO_NAME}: dataset jintran0501/ai-live-infinitetalk-input → New Version, tải {AUDIO_NAME} lên "
        "(giữ các file cũ), rồi gắn lại bản mới nhất của dataset."
    )
frame_dirs = sorted({path.parent for path in frame_paths})
if len(frame_dirs) > 1:
    print(f"Có {len(frame_dirs)} thư mục khung, dùng thư mục đầu: {frame_dirs[0]}")
frame_paths = sorted(frame_dirs[0].glob(FRAME_GLOB))
source_audio = audio_candidates[0]

frames = [cv2.imread(str(path)) for path in frame_paths]
height, width = frames[0].shape[:2]
n_frames = len(frames)


def probe_seconds(path: Path) -> float:
    return float(run("ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(path)))


full_audio = WORK_DIR / "train.wav"
run("ffmpeg", "-y", "-v", "error", "-i", str(source_audio), "-ac", "1", "-ar", "16000", str(full_audio))
audio_seconds = probe_seconds(full_audio)
print(f"Khung: {n_frames} ({width}×{height}) từ {frame_dirs[0]}")
print(f"Tiếng: {source_audio} ({audio_seconds:.1f} s, khoảng {int(audio_seconds * FPS)} khung)")
if audio_seconds < 60:
    print("Cảnh báo: tiếng dưới 1 phút. Tác giả Ultralight khuyên 3–5 phút.")

## 5. Khung cắt mặt

Script chính thức lấy 68 điểm mặt bằng DWPose (cần mmcv/mmpose). Ở đây dùng gói `face-alignment`: cùng 68 điểm theo thứ tự iBUG, nên công
thức khung cắt giữ y như `get_landmark_and_bbox` của MuseTalk (ADR-063). Ảnh `crop_check.png` vẽ khung cắt và điểm mặt để kiểm.

In [ ]:
fa_types = face_alignment.LandmarksType
fa = face_alignment.FaceAlignment(getattr(fa_types, "TWO_D", None) or fa_types._2D, flip_input=False, device="cuda")


def musetalk_bbox(landmarks: np.ndarray, detection: np.ndarray) -> tuple[int, int, int, int]:
    """Same box rule as MuseTalk's get_landmark_and_bbox with bbox_shift=0 (v1.5)."""
    lm = landmarks.astype(np.int32)
    half_face_y = lm[29][1]
    half_face_dist = np.max(lm[:, 1]) - half_face_y
    upper_bond = max(0, half_face_y - half_face_dist)
    x1, y1, x2, y2 = int(np.min(lm[:, 0])), int(upper_bond), int(np.max(lm[:, 0])), int(np.max(lm[:, 1]))
    if y2 - y1 <= 0 or x2 - x1 <= 0 or x1 < 0:
        # MuseTalk falls back to the detector box when the landmark box is unusable.
        return tuple(int(v) for v in detection[:4])
    return x1, y1, x2, y2


coords: list[tuple[int, int, int, int]] = []
landmarks_all: list[np.ndarray] = []
started = time.time()
for index, frame in enumerate(frames):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = fa.get_landmarks_from_image(rgb, return_bboxes=True)
    landmarks, _, detections = result if isinstance(result, tuple) else (result, None, None)
    if not landmarks:
        raise SystemExit(f"Khung {index}: không thấy mặt.")
    # The host is the only face; take the largest detection if a poster or product label fools the detector.
    best = max(range(len(landmarks)), key=lambda i: np.ptp(landmarks[i][:, 0]) * np.ptp(landmarks[i][:, 1]))
    landmarks_all.append(landmarks[best])
    coords.append(musetalk_bbox(landmarks[best], detections[best]))
landmark_seconds = time.time() - started

check = frames[0].copy()
x1, y1, x2, y2 = coords[0]
cv2.rectangle(check, (x1, y1), (x2, min(y2 + EXTRA_MARGIN, height)), (0, 255, 0), 2)
for x, y in landmarks_all[0].astype(int):
    cv2.circle(check, (int(x), int(y)), 2, (0, 0, 255), -1)
cv2.imwrite(str(OUT_DIR / "crop_check.png"), check)
widths = [c[2] - c[0] for c in coords]
print(
    f"Điểm mặt {n_frames} khung trong {landmark_seconds:.1f} s; rộng khung cắt {min(widths)}–{max(widths)} px "
    f"(MuseTalk phóng lên 256 px)"
)

## 6. Lồng miệng cả file tiếng, ghi thẳng ra video

Khung được ghép xong là ghi ngay vào ffmpeg qua pipe, không giữ trong RAM (khoảng 7.300 khung). Khung nền chạy tới rồi lùi như
script chính thức, nên đầu và vai chuyển động liền mạch.

In [ ]:
os.chdir(MUSETALK_DIR)  # MuseTalk loads models/ by relative path
sys.path.insert(0, str(MUSETALK_DIR))

from musetalk.utils.audio_processor import AudioProcessor  # noqa: E402
from musetalk.utils.blending import get_image  # noqa: E402
from musetalk.utils.face_parsing import FaceParsing  # noqa: E402
from musetalk.utils.utils import datagen, load_all_model  # noqa: E402
from transformers import WhisperModel  # noqa: E402

device = torch.device("cuda")
vae, unet, pe = load_all_model(
    unet_model_path=str(MODELS_DIR / "musetalkV15" / "unet.pth"),
    vae_type="sd-vae",
    unet_config=str(MODELS_DIR / "musetalkV15" / "musetalk.json"),
    device=device,
)
if USE_FP16:
    pe = pe.half()
    vae.vae = vae.vae.half()
    unet.model = unet.model.half()
pe = pe.to(device)
vae.vae = vae.vae.to(device)
unet.model = unet.model.to(device)
weight_dtype = unet.model.dtype
timesteps = torch.tensor([0], device=device)

audio_processor = AudioProcessor(feature_extractor_path=str(MODELS_DIR / "whisper"))
whisper = WhisperModel.from_pretrained(str(MODELS_DIR / "whisper")).to(device=device, dtype=weight_dtype).eval()
whisper.requires_grad_(False)
face_parser = FaceParsing(left_cheek_width=LEFT_CHEEK_WIDTH, right_cheek_width=RIGHT_CHEEK_WIDTH)


def crop_box(coord: tuple[int, int, int, int]) -> tuple[int, int, int, int]:
    x1, y1, x2, y2 = coord
    return x1, y1, x2, min(y2 + EXTRA_MARGIN, height)  # v1.5 extends the chin margin


with torch.no_grad():
    input_latents = []
    for coord, frame in zip(coords, frames, strict=True):
        x1, y1, x2, y2 = crop_box(coord)
        crop = cv2.resize(frame[y1:y2, x1:x2], (256, 256), interpolation=cv2.INTER_LANCZOS4)
        input_latents.append(vae.get_latents_for_unet(crop))


# One fixed mouth window from the median landmarks for the close-up sheet.
mouth_points = np.concatenate([lm[48:68] for lm in landmarks_all])
center = np.median(mouth_points, axis=0)
mouth_width = float(np.median([np.ptp(lm[48:68, 0]) for lm in landmarks_all]))
half = int(mouth_width * 0.9)
mx1, my1 = max(0, int(center[0]) - half), max(0, int(center[1]) - int(half * 0.75))
mx2, my2 = min(width, int(center[0]) + half), min(height, int(center[1]) + int(half * 0.75))


def mouth(image: np.ndarray) -> np.ndarray:
    crop = image[my1:my2, mx1:mx2]
    scale = MOUTH_CLOSEUP_PX / crop.shape[0]
    return cv2.resize(crop, (int(crop.shape[1] * scale), MOUTH_CLOSEUP_PX), interpolation=cv2.INTER_CUBIC)


sheet_start = int(PREVIEW_START_S * FPS)
SHEET_PICKS = set(range(sheet_start, sheet_start + 28 * 3, 3))  # 28 tiles, 3 frames apart, after the silence
sheet_tiles: dict[int, np.ndarray] = {}
video_path = OUT_DIR / VIDEO_NAME


@torch.no_grad()
def dub_to_video(audio_path: Path, path: Path) -> dict:
    """Dub audio_path over the frames (forward then backward) and stream the result into an mp4 with that audio."""
    order = list(range(n_frames)) + list(range(n_frames))[::-1]
    features, librosa_length = audio_processor.get_audio_feature(str(audio_path))
    chunks = audio_processor.get_whisper_chunk(
        features,
        device,
        weight_dtype,
        whisper,
        librosa_length,
        fps=FPS,
        audio_padding_length_left=AUDIO_PADDING_LEFT,
        audio_padding_length_right=AUDIO_PADDING_RIGHT,
    )
    total = len(chunks)
    latents = [input_latents[i] for i in order]
    log = (WORK_DIR / "ffmpeg_video.log").open("w")
    writer = subprocess.Popen(
        [
            "ffmpeg",
            "-y",
            "-v",
            "error",
            "-f",
            "rawvideo",
            "-pix_fmt",
            "bgr24",
            "-s",
            f"{width}x{height}",
            "-r",
            str(FPS),
            "-i",
            "-",
            "-i",
            str(audio_path),
            "-map",
            "0:v",
            "-map",
            "1:a",
            "-t",
            f"{total / FPS:.3f}",
            "-vf",
            "pad=ceil(iw/2)*2:ceil(ih/2)*2",
            "-c:v",
            "libx264",
            "-crf",
            "16",
            "-pix_fmt",
            "yuv420p",
            "-c:a",
            "aac",
            "-b:a",
            "128k",
            str(path),
        ],
        stdin=subprocess.PIPE,
        stdout=log,
        stderr=log,
    )
    model_seconds = blend_seconds = 0.0
    written = 0
    started_all = time.time()
    for whisper_batch, latent_batch in datagen(chunks, latents, batch_size=BATCH_SIZE, delay_frame=0, device=device):
        torch.cuda.synchronize()
        started = time.time()
        audio_features = pe(whisper_batch.to(device))
        prediction = unet.model(
            latent_batch.to(device, dtype=weight_dtype), timesteps, encoder_hidden_states=audio_features
        ).sample
        faces = vae.decode_latents(prediction)
        torch.cuda.synchronize()
        model_seconds += time.time() - started

        started = time.time()
        for face in faces:
            src = order[written % len(order)]
            x1, y1, x2, y2 = crop_box(coords[src])
            face = cv2.resize(face.astype(np.uint8), (x2 - x1, y2 - y1))
            frame = get_image(frames[src].copy(), face, [x1, y1, x2, y2], mode=PARSING_MODE, fp=face_parser)
            if written in SHEET_PICKS:
                sheet_tiles[written] = mouth(frame)
            writer.stdin.write(np.ascontiguousarray(frame, dtype=np.uint8).tobytes())
            written += 1
        blend_seconds += time.time() - started
        if written % 1000 < BATCH_SIZE:
            print(f"{written}/{total} khung ({time.time() - started_all:.0f} s)", flush=True)
    writer.stdin.close()
    writer.wait()
    log.close()
    if writer.returncode != 0:
        tail = (WORK_DIR / "ffmpeg_video.log").read_text()[-3000:]
        raise RuntimeError(f"ffmpeg lỗi (mã {writer.returncode}):\n{tail}")
    return {
        "frames": written,
        "seconds": round(written / FPS, 2),
        "model_seconds": round(model_seconds, 1),
        "blend_seconds": round(blend_seconds, 1),
        "model_fps": round(written / model_seconds, 2),
        "end_to_end_fps": round(written / (time.time() - started_all), 2),
    }


dub_stats = dub_to_video(full_audio, video_path)
print(
    f"{VIDEO_NAME}: {dub_stats['frames']} khung ({dub_stats['seconds']} s), UNet+VAE {dub_stats['model_fps']} khung/s, "
    f"cả quá trình {dub_stats['end_to_end_fps']} khung/s"
)
print(f"VRAM đỉnh: {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")

## 7. Kiểm video, đoạn xem trước, ảnh ghép miệng, `run.json`

In [ ]:
os.chdir("/kaggle/working")
FONT = cv2.FONT_HERSHEY_SIMPLEX

# The training notebook needs 25 fps video with sound; check the file it will read, not the numbers above.
probe_args = ["-v", "error", "-print_format", "json", "-show_streams", "-show_format", str(video_path)]
info = json.loads(run("ffprobe", *probe_args))
video_stream = next(s for s in info["streams"] if s["codec_type"] == "video")
has_audio = any(s["codec_type"] == "audio" for s in info["streams"])
if video_stream["r_frame_rate"] != f"{FPS}/1" or not has_audio:
    raise RuntimeError(f"Video sai: {video_stream['r_frame_rate']} khung/giây, có tiếng = {has_audio}.")
video_seconds = round(float(info["format"]["duration"]), 2)
if abs(video_seconds - audio_seconds) > 1.0:
    print(f"Cảnh báo: video {video_seconds} s, tiếng {audio_seconds:.2f} s.")

run(
    "ffmpeg",
    "-y",
    "-v",
    "error",
    "-ss",
    f"{PREVIEW_START_S:.2f}",
    "-i",
    str(video_path),
    "-t",
    f"{PREVIEW_SECONDS:.2f}",
    "-c:v",
    "libx264",
    "-crf",
    "18",
    "-pix_fmt",
    "yuv420p",
    "-c:a",
    "aac",
    "-b:a",
    "128k",
    str(OUT_DIR / "preview.mp4"),
)

tiles = []
for index in sorted(sheet_tiles):
    tile = cv2.resize(sheet_tiles[index], None, fx=0.5, fy=0.5)
    cv2.putText(tile, f"{index / FPS:.2f}s", (6, 22), FONT, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
    tiles.append(tile)
while len(tiles) % 7:
    tiles.append(np.zeros_like(tiles[0]))
rows = [np.hstack(tiles[i : i + 7]) for i in range(0, len(tiles), 7)]
cv2.imwrite(str(OUT_DIR / "mouth_sheet.png"), np.vstack(rows))

run_info = {
    "adr": "ADR-064",
    "teacher": TEACHER,
    "character_id": CHARACTER_ID,
    "run_name": RUN_NAME,
    "created_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    "gpu": GPU_NAME,
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "diffusers": diffusers.__version__,
        "face_alignment": face_alignment.__version__,
        "musetalk_commit": MUSETALK_COMMIT,
    },
    "weights": weights_info,
    "inputs": {
        "frames_dir": str(frame_dirs[0]),
        "frames": n_frames,
        "frame_size": [width, height],
        "frame_order": "forward then backward",
        "audio": str(source_audio),
        "audio_sha256": sha256(source_audio),
        "audio_seconds": round(audio_seconds, 2),
    },
    "settings": {
        "fps": FPS,
        "batch_size": BATCH_SIZE,
        "extra_margin": EXTRA_MARGIN,
        "cheek_width": [LEFT_CHEEK_WIDTH, RIGHT_CHEEK_WIDTH],
        "parsing_mode": PARSING_MODE,
        "audio_padding": [AUDIO_PADDING_LEFT, AUDIO_PADDING_RIGHT],
        "fp16": USE_FP16,
        "landmarks": "face-alignment 68 pts (replaces DWPose)",
    },
    "crop_width_px": [min(widths), max(widths)],
    "output": {
        "file": VIDEO_NAME,
        "sha256": sha256(video_path),
        "seconds": video_seconds,
        "frames": dub_stats["frames"],
    },
    "timing": {
        "landmarks_seconds": round(landmark_seconds, 1),
        "dub": dub_stats,
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
        "notebook_seconds": None,
    },
}
(OUT_DIR / "run.json").write_text(json.dumps(run_info, ensure_ascii=False, indent=2))
for path in sorted(OUT_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size / 2**20:.1f} MB")

## 8. Bước tiếp

1. Xem `preview.mp4` có tiếng. Miệng mở quá ít thì thử `EXTRA_MARGIN` 15–20 rồi chạy lại (miễn phí, khoảng 30 phút).
2. Mở notebook `ai-host-train-ultralight-demo-host-02`, đặt `TEACHER = "musetalk"`, Add Input → **Notebook Output** → notebook
   này, rồi Save & Run All.
3. Trước buổi demo: tạo video bằng InfiniteTalk (`RUN_NAME = "train_v1"`), đổi `TEACHER = "infinitetalk"` trong notebook huấn luyện
   và huấn luyện lại. Hai model nằm ở hai thư mục riêng (`ultralight_out/<CHARACTER_ID>/<TEACHER>/`), không ghi đè nhau.